In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv


In [2]:
train_df = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')
train_df.head(3)

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Age                      630000 non-null  int64  
 2   Sex                      630000 non-null  int64  
 3   Chest pain type          630000 non-null  int64  
 4   BP                       630000 non-null  int64  
 5   Cholesterol              630000 non-null  int64  
 6   FBS over 120             630000 non-null  int64  
 7   EKG results              630000 non-null  int64  
 8   Max HR                   630000 non-null  int64  
 9   Exercise angina          630000 non-null  int64  
 10  ST depression            630000 non-null  float64
 11  Slope of ST              630000 non-null  int64  
 12  Number of vessels fluro  630000 non-null  int64  
 13  Thallium                 630000 non-null  int64  
 14  Hear

In [4]:
train_df = pd.get_dummies(train_df , columns = ['Heart Disease'] , drop_first = True , dtype = int)

In [5]:
train_df = train_df.drop(columns = ['id'] , axis = 1)
train_df.head(3)

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease_Presence
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,1
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,0
2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,0


In [6]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Age                      630000 non-null  int64  
 1   Sex                      630000 non-null  int64  
 2   Chest pain type          630000 non-null  int64  
 3   BP                       630000 non-null  int64  
 4   Cholesterol              630000 non-null  int64  
 5   FBS over 120             630000 non-null  int64  
 6   EKG results              630000 non-null  int64  
 7   Max HR                   630000 non-null  int64  
 8   Exercise angina          630000 non-null  int64  
 9   ST depression            630000 non-null  float64
 10  Slope of ST              630000 non-null  int64  
 11  Number of vessels fluro  630000 non-null  int64  
 12  Thallium                 630000 non-null  int64  
 13  Heart Disease_Presence   630000 non-null  int64  
dtypes: f

In [7]:
y = train_df['Heart Disease_Presence']
x = train_df.drop(columns = ['Heart Disease_Presence'] , axis = 1)

In [8]:
x_train , x_val , y_train , y_val = train_test_split(x , y ,test_size = 0.3 , random_state = 58)

In [9]:
lr = xgb.XGBClassifier(
    n_estimators=1000,       # Daha fazla ağaç
    learning_rate=0.01,      # Daha küçük adımlar (overfitting'i engeller)
    max_depth=6,             # Ağaç derinliği
    subsample=0.8,           # Her adımda verinin %80'ini rastgele seç
    colsample_bytree=0.8,    # Her adımda özelliklerin %80'ini seç
    n_jobs=-1,               # Tüm çekirdekleri kullan
    random_state=58,
    early_stopping_rounds=50, # 50 adım boyunca gelişmezse durur
)
model = lr.fit(
    x_train, y_train,
    eval_set=[(x_val, y_val)],  
    verbose=False              # Kalabalık yapmasın diye False kalsın
)

In [10]:
print(f"Model Accuracy Score: {model.score(x_val, y_val)}")

Model Accuracy Score: 0.8874550264550265


In [11]:
test_df = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')
test_df.head(3)

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3
1,630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3
2,630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7


In [12]:
# 1. Test verisini hazırla
# Önce ID'leri bir değişkende saklıyoruz
test_ids = test_df['id']

# Train'de yaptığın gibi ID sütununu buradan da atıyoruz
# Eğer train'de başka bir sütun sildiysen (örneğin target), burada sadece ID'yi atman yeterli
x_test_final = test_df.drop(columns=['id'], axis=1)

# 2. Tahminleri al
# NOT: predict() yerine predict_proba() kullanarak olasılık alıyoruz (Kaggle bunu sever)
# [:, 1] ifadesi 'Kalp Hastalığı Var' (Presence) olasılığını seçer
test_predictions = model.predict_proba(x_test_final)[:, 1]

# 3. Submission dosyasını oluştur
submission = pd.DataFrame({
    'id': test_ids,
    'Heart Disease': test_predictions
})

# 4. CSV olarak kaydet
submission.to_csv('submission.csv', index=False)

print("İşlem tamam! 'submission.csv' dosyasını Kaggle'a yükleyebilirsin.")

İşlem tamam! 'submission.csv' dosyasını Kaggle'a yükleyebilirsin.


In [13]:
sub_df = pd.read_csv('submission.csv')
sub_df.head(3)

,id,Heart Disease
0,630000,0.940995
1,630001,0.008101
2,630002,0.981338
